In [72]:
import matplotlib.pyplot as plt
import seaborn as sns
import sys
sys.path.append('../src/')
import features.ptype_prepare_data as pp
import models.ptype_run_preds as rp
import models.score_classifier as score
import viz.ptype_visualize as viz
import pandas as pd
import pickle
import os
import boto3
from datetime import datetime
from catboost import CatBoostClassifier
from sklearn.model_selection import cross_val_score, GridSearchCV, RandomizedSearchCV
from sklearn.metrics import roc_curve, roc_auc_score, precision_recall_curve, f1_score, precision_score, recall_score, confusion_matrix, ConfusionMatrixDisplay

In [85]:
DOWNLOAD_DATA = False
SS0_PROFILE_NAME = 'rognstad-dev'

In [86]:
if DOWNLOAD_DATA:
    boto3.setup_default_session(profile_name=SS0_PROFILE_NAME)
    bucket_name = 'restoration-monitoring'
    folder_prefix = 'plantation-mapping/data/train/'
    data_prefix_list = ['slope', 's1', 's2', 'dem', 'features-ckpt-2023-02-09', 'labels']
    local_prefix = '../data/train/'
    conn = boto3.client('s3')
    for prefix in data_prefix_list:
        print(prefix)
        prefix_string = f'{folder_prefix}train-{prefix}/'
        outpath = f'{local_prefix}/train-{prefix}/'
        file_list = []
        for object in conn.list_objects_v2(Bucket=bucket_name,Prefix=prefix_string)['Contents']:
            file_list.append(object['Key'])
    
        for file in file_list:
            conn.download_file(Bucket=bucket_name, Key=file, Filename=(outpath + os.path.basename(file)))

labels


In [100]:
ceo_summary = pd.read_excel('../data/Training_Data.xlsx', sheet_name='CEO_Surveys', header=1)
ceo_summary = ceo_summary[ceo_summary['CEO Survey'].notna()]

In [101]:
ceo_summary[ceo_summary['CEO Survey'].notna()]

,CEO Survey,Year Collected,Data Source,Municipality,Country,Purpose,Vegetation categories (observed),Plots in Polygons,Plots outside Polygons,Final plot count,Classes,Non plantation (0),Plantation (for mult-class this is mono) (1),Agroforestry (2),Plot ID,Use?,Notes
0,plantations-train-v00,NaN,SDPT,Colon,Honduras,prototyping,NaN,7.0,0.0,NaN,Binary,NaN,NaN,NaN,NaN,yes,NaN
1,plantations-train-v01,NaN,SDPT,Colon,Honduras,prototyping,NaN,8.0,0.0,NaN,Binary,NaN,NaN,NaN,NaN,yes,NaN
2,plantations-train-v02,NaN,SDPT,Copan,Honduras,prototyping,coffee agroforestry,100.0,0.0,NaN,Binary,NaN,NaN,NaN,NaN,no,NaN
3,plantations-train-v03,NaN,SDPT,Multiple,Guatemala,Pilot #2,"oil palm, orchards",100.0,0.0,100.0,Binary,NaN,NaN,NaN,0-99,yes,NaN
4,plantations-train-v04,NaN,SDPT,Multiple,Guatemala,Pilot #2,non plantation,0.0,66.0,66.0,Binary,NaN,NaN,NaN,4000-40099 (not consecutive),yes,NaN
5,plantations-train-v05,NaN,IIASA,Multiple,"Brazil, Costa Rica, Ivory Coast, Ghana, Guatemala",prototyping,Short Rotation Plantations for Timber,NaN,100.0,NaN,Binary,NaN,NaN,NaN,NaN,no,NaN
6,plantations-train-v06,NaN,IIASA,Multiple,"Brazil, Costa Rica, Ivory Coast, Ghana, Guatem...",prototyping,Oil palm,NaN,101.0,101.0,Binary,NaN,NaN,NaN,40100-40200,yes,NaN
7,plantations-train-v07,NaN,IIASA,Multiple,"Brazil, Costa Rica, Ivory Coast, Ghana, Guatem...",prototyping,Agroforestry,NaN,101.0,NaN,Binary,NaN,NaN,NaN,NaN,no,NaN
8,plantations-train-v08_OLD,NaN,SDPT,Multiple,"Ghana, Ivory Coast",Pilot #1,"oil palm, rubber, cashew, agroforestry",100.0,125.0,213.0,Binary,25095.0,16653.0,NaN,8001-8225 (not consecutive),yes,Agroforestry is labeled as 0
9,plantations-train-v08,NaN,SDPT,Multiple,"Ghana, Ivory Coast",Pilot #1,"oil palm, rubber, cashew, agroforestry",100.0,125.0,177.0,Multi,9179.0,12247.0,13266.0,8000,yes,NaN


In [9]:
X, y = pp.create_xy(['v08'], 
                    classes='binary', 
                    drop_feats=False,
                    data_dir ='../data', 
                    verbose=False)


0.0 plots labeled unknown were dropped from v08.
warning needs to be updated
Plot id 08001 has no cloud free imagery and will be removed.
Plot id 08002 has no cloud free imagery and will be removed.
Plot id 08003 has no cloud free imagery and will be removed.
Plot id 08004 has no cloud free imagery and will be removed.
Plot id 08005 has no cloud free imagery and will be removed.
Plot id 08006 has no cloud free imagery and will be removed.
Plot id 08007 has no cloud free imagery and will be removed.
Plot id 08008 has no cloud free imagery and will be removed.
Plot id 08009 has no cloud free imagery and will be removed.
Plot id 08010 has no cloud free imagery and will be removed.
Plot id 08011 has no cloud free imagery and will be removed.
Plot id 08012 has no cloud free imagery and will be removed.
Plot id 08013 has no cloud free imagery and will be removed.
Plot id 08014 has no cloud free imagery and will be removed.
Plot id 08015 has no cloud free imagery and will be removed.
Plot id 

0it [00:00, ?it/s]

Class count {}
